<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_04_time_aware_data_splitting/stage_04_time_aware_data_splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_04_time_aware_data_splitting**

## Introducción y Resumen

Esta notebook tiene como objetivo dividir el dataset MNQ en conjuntos de entrenamiento, validación y prueba, asegurando una partición aleatoria, reproducible y estructuralmente consistente. Con ello se dejan listos los datos para el entrenamiento y evaluación de los modelos predictivos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente (mnq_technical_indicators y mnq_alpha_factors) y se muestra un resumen de la información del dataset MNQ.

1. Carga de datos

    Se importa el dataset procesado con features técnicos y alpha factors. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.

2. Análisis del dataset `mnq_model`

    Se revisa la estructura del dataset (filas, columnas, tipos de datos), se busca los valores NaNs y se verifica la distribución temporal de los registros.

3. Definición de parámetros de división

    En este punto se define la estrategia de partición del dataset: se toma un 70% de los días para entrenamiento, y el 30% restante se divide en partes iguales para validación y prueba. De esta manera, el modelo cuenta con suficientes datos para aprender, mientras que se reservan bloques temporales separados para ajustar parámetros y evaluar el rendimiento final sin fugas de información.

4. Selección aleatoria de días.

    Este punto busca garantizar que la partición de los datos sea representativa y no esté sesgada por la secuencia temporal. Al asignar los días de forma aleatoria —aunque de manera reproducible— se evita que los conjuntos queden condicionados por períodos específicos del mercado (por ejemplo, tendencias prolongadas o alta volatilidad en ciertos meses). Así, cada subconjunto refleja mejor la diversidad del dataset y se obtiene una evaluación más robusta del modelo.

5. Generación de datasets `mnq_train`, `mnq_test` y `mnq_valid`

    En este punto se crean los datasets mnq_train, mnq_valid y mnq_test, manteniendo homogeneidad en estructura (301 registros por día, de 09:30 a 14:30) y sin solapamiento entre conjuntos. Esto asegura consistencia en el entrenamiento, validación y prueba del modelo.

## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [ ]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [2]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr

import os
import json
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

#from ta.momentum import ROCIndicator



In [3]:
# ============================================================
# Paths / IO (via env o defaults)
# ============================================================

DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

IN_PARQUET_DELTA_60 = Path(os.environ.get("IN_PARQUET_DELTA_60", "data/features/mnq_delta_60.parquet"))
IN_PARQUET_DELTA_90 = Path(os.environ.get("IN_PARQUET_DELTA_90", "data/features/mnq_delta_90.parquet"))

##############

OUT_SPLITS = Path(os.environ.get("OUT_PARQUET", "data/splits/splits.json"))

OUT_PARQUET_DELTA_60_TRAIN = Path(os.environ.get("OUT_PARQUET_DELTA_60_TRAIN", "data/splits/mnq_delta_60_train.parquet"))
OUT_PARQUET_DELTA_60_VALID = Path(os.environ.get("OUT_PARQUET_DELTA_60_VALID", "data/splits/mnq_delta_60_valid.parquet"))
OUT_PARQUET_DELTA_60_TEST = Path(os.environ.get("OUT_PARQUET_DELTA_60_TEST", "data/splits/mnq_delta_60_test.parquet"))

OUT_PARQUET_DELTA_90_TRAIN = Path(os.environ.get("OUT_PARQUET_DELTA_90_TRAIN", "data/splits/mnq_delta_90_train.parquet"))
OUT_PARQUET_DELTA_90_VALID = Path(os.environ.get("OUT_PARQUET_DELTA_90_VALID", "data/splits/mnq_delta_90_valid.parquet"))
OUT_PARQUET_DELTA_90_TEST = Path(os.environ.get("OUT_PARQUET_DELTA_90_TEST", "data/splits/mnq_delta_90_test.parquet"))

# PARA EL NOTEBOOK:

IN_PARQUET_DELTA_60 = DRIVE_DIR / IN_PARQUET_DELTA_60
IN_PARQUET_DELTA_90 = DRIVE_DIR / IN_PARQUET_DELTA_90


OUT_PARQUET_DELTA_60_TRAIN = DRIVE_DIR / OUT_PARQUET_DELTA_60_TRAIN
OUT_PARQUET_DELTA_60_VALID = DRIVE_DIR / OUT_PARQUET_DELTA_60_VALID
OUT_PARQUET_DELTA_60_TEST = DRIVE_DIR / OUT_PARQUET_DELTA_60_TEST

OUT_PARQUET_DELTA_90_TRAIN = DRIVE_DIR / OUT_PARQUET_DELTA_90_TRAIN
OUT_PARQUET_DELTA_90_VALID = DRIVE_DIR / OUT_PARQUET_DELTA_90_VALID
OUT_PARQUET_DELTA_90_TEST = DRIVE_DIR / OUT_PARQUET_DELTA_90_TEST





## **1. Carga de datos**

### 1.1. Carga de dataset `mnq_delta_*.parquet`




In [4]:
def _ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

def load_mnq_parquet(path: Path):
    os.path.exists(path)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(path)
    return mnq_parquet

### 1.2. Información de dataset `mnq_`



In [5]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")



## **1.3. Carga de mnq e información**




In [6]:
mnq_delta_60 = load_mnq_parquet(IN_PARQUET_DELTA_60)
info_mnq_delta_60 = mnq_dataset_info(mnq_delta_60, name="mnq_delta_60", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_delta_60)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_delta_60
Shape: (700595, 14)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10', 'delta_60']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 05:30:00-05:00  ->  2025-06-13 14:30:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 330, 'max_minute_of_day': 870}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 10:30:00+00:00  ->  2025-06-13 18:30:00+00:00


In [7]:
mnq_delta_90 = load_mnq_parquet(IN_PARQUET_DELTA_90)
info_mnq_delta_90 = mnq_dataset_info(mnq_delta_90, name="mnq_delta_90", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_delta_90)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_delta_90
Shape: (700595, 14)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10', 'delta_90']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 05:30:00-05:00  ->  2025-06-13 14:30:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 330, 'max_minute_of_day': 870}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 10:30:00+00:00  ->  2025-06-13 18:30:00+00:00


# **2. Análisis de datasets**

## **2.0. Funciones**

### Función para contar NaN en dataset

In [8]:
def nan_count(df: pd.DataFrame, mnq: str) -> None:
    """
    Verifica la existencia de NaN por día y por columna.
    - Si existen NaN, muestra únicamente las columnas afectadas.
    - Si no existen, muestra un mensaje indicando que no hay NaNs.
    """

    # Conteo de NaN por día y columna
    daily_nan_counts = (
        df.groupby("date")
        .apply(lambda x: x.isna().sum())
    )

    # Identificar columnas con al menos un NaN en cualquier día
    cols_with_nan = daily_nan_counts.columns[
        (daily_nan_counts > 0).any(axis=0)
    ].tolist()

    if cols_with_nan:
        print("Columnas con valores NaN:")
        for col in cols_with_nan:
            total_nans = df[col].isna().sum()
            print(f" - {col}: {total_nans} NaNs")
    else:
        print(f"{mnq} sin valores NaNs")

### Función para detectar saltos temporales (gaps)

In [9]:
import pandas as pd

def detectar_gaps(df: pd.DataFrame, mnq: str, gap_minutes: int = 1):
    """
    Verifica si existen saltos mayores al intervalo esperado (por defecto 1 minuto)
    entre registros consecutivos dentro de cada día, en un DataFrame con índice tipo DatetimeIndex.

    Omite el primer registro de cada día.

    Parámetros:
    - df: DataFrame con índice datetime.
    - mnq_delta_h: nombre/identificador del dataset (para mensajes).
    - gap_minutes: tamaño esperado del intervalo en minutos (por defecto 1).

    Retorna:
    - Lista de índices donde se detectaron diferencias mayores al intervalo esperado.
      (por día se guarda un Index con los timestamps irregulares)
    """
    df = df.copy()
    df["time_diff"] = df.index.to_series().diff()

    base_time_diff = pd.Timedelta(minutes=gap_minutes)
    problem_indices = []

    for date, group in df.groupby(df.index.date):
        time_diff = group["time_diff"].iloc[1:]  # omite el primer registro del día
        incorrect_indices = time_diff[time_diff != base_time_diff].index
        if len(incorrect_indices) > 0:
            problem_indices.append(incorrect_indices)

    if problem_indices:
        print(f"{mnq} con gaps")
        print(f"Se encontraron problemas en {sum(len(x) for x in problem_indices)} registros con diferencias irregulares.\n")

        # Conteo por fecha
        conteos = df.groupby(df.index.date).size()

        for day_idx in problem_indices:
            idx = day_idx[0]  # primer timestamp irregular del día
            diff = df.loc[idx, "time_diff"]
            date = idx.date()
            count = conteos[date]
            print(f"\t{idx} -> Diferencia: {diff} | # Registros: {count}")
    else:
        print(f"{mnq} sin gaps")

    #return problem_indices


### Función para aplicar análisis

In [10]:
def check_df(df, df_name = str):
  nan_count(df, df_name)
  detectar_gaps(df, df_name)

## **2.1. Aplicación de análisis**

In [11]:
check_df (mnq_delta_60, 'mnq_delta_60')

mnq_delta_60 sin valores NaNs
mnq_delta_60 sin gaps


In [12]:
check_df (mnq_delta_90, 'mnq_delta_90')

mnq_delta_90 sin valores NaNs
mnq_delta_90 sin gaps


# **3. Definición de parámetros de división**

Para dividir el dataset en subconjuntos, se utiliza la siguiente estrategia:

- 70% de los días se asignan al conjunto de entrenamiento (train).

- El 30% restante se reparte de manera equitativa entre los conjuntos de validación (valid) y prueba (test).

Esto garantiza que el modelo disponga de la mayor parte de los datos para aprender patrones, mientras que las particiones de validación y prueba permiten ajustar hiperparámetros y evaluar el rendimiento fuera de muestra.

De esta forma, se asegura un esquema de división temporalmente consistente, sin solapamiento entre conjuntos.

#**4. Partición del dataset respetando causalidad temporal**

En este punto, la división del dataset se realiza respetando el orden cronológico de los días, con el objetivo de preservar la causalidad temporal y evitar cualquier forma de data leakage en la evaluación del modelo.

Para ello, se extraen los días únicos presentes en el dataset y se ordenan cronológicamente. A continuación, se asignan los primeros días al conjunto de entrenamiento, los días intermedios al conjunto de validación y los días más recientes al conjunto de prueba, de acuerdo con las proporciones definidas (70 % / 15 % / 15 %).

Este procedimiento garantiza que el modelo sea entrenado exclusivamente con información pasada y evaluado sobre datos futuros, manteniendo la integridad intradía de cada jornada y proporcionando una estimación realista de su capacidad de generalización.

In [13]:
# Obtener días únicos ordenados
unique_days_delta_60 = pd.Index(sorted(mnq_delta_60["date"].unique()))
unique_days_delta_90 = pd.Index(sorted(mnq_delta_90["date"].unique()))

# Dataset de referencia
reference = unique_days_delta_60

datasets = {
    "delta_60": unique_days_delta_60,
    "delta_90": unique_days_delta_90,

}

all_ok = True

for name, days in datasets.items():
    if not reference.equals(days):
        all_ok = False
        missing_in_current = reference.difference(days)
        extra_in_current = days.difference(reference)

        print(f"{name} NO coincide con delta_60")
        if len(missing_in_current) > 0:
            print(f"  - Faltan días: {list(missing_in_current)}")
        if len(extra_in_current) > 0:
            print(f"  - Días adicionales: {list(extra_in_current)}")
        print()

if all_ok:
    print("Todos los datasets tienen exactamente los mismos días.")

Todos los datasets tienen exactamente los mismos días.


In [14]:
# Tomamos delta_60 como referencia (ya validaste que todos coinciden)
unique_days = pd.Index(sorted(mnq_delta_60["date"].unique()))

n_total = len(unique_days)

n_train = int(n_total * 0.7)
n_valid = int(n_total * 0.15)
n_test  = n_total - n_train - n_valid

# Dividir días en orden temporal
train_days = unique_days[:n_train]
val_days   = unique_days[n_train:n_train + n_valid]
test_days  = unique_days[n_train + n_valid:]

print(f"Train days: {len(train_days)}")
print(f"Valid days: {len(val_days)}")
print(f"Test days : {len(test_days)}")


Train days: 906
Valid days: 194
Test days : 195


In [15]:
print("Train days:")
print(f"  Desde: {train_days[0]}")
print(f"  Hasta: {train_days[-1]}")

print("\nValidation days:")
print(f"  Desde: {val_days[0]}")
print(f"  Hasta: {val_days[-1]}")

print("\nTest days:")
print(f"  Desde: {test_days[0]}")
print(f"  Hasta: {test_days[-1]}")

Train days:
  Desde: 2020-01-02
  Hasta: 2023-10-30

Validation days:
  Desde: 2023-10-31
  Hasta: 2024-08-21

Test days:
  Desde: 2024-08-22
  Hasta: 2025-06-13


# **5. Generación de datasets `mnq_train`, `mnq_test` y `mnq_valid`**

En este paso generamos los datasets finales para cada subconjunto: `mnq_train`, `mnq_valid` y `mnq_test`. La asignación se realiza filtrando los días correspondientes a cada conjunto, lo que asegura que no exista solapamiento entre ellos.

In [24]:
def comprobar_orden_datasets_base(**datasets):
    print("=" * 80)
    print("VALIDACIÓN ORDEN TEMPORAL - DATASETS BASE")
    print("=" * 80)

    for name, df in datasets.items():
        ordenado = df[["date", "minute_of_day"]].reset_index(drop=True).equals(
            df.sort_values(["date", "minute_of_day"])[["date", "minute_of_day"]].reset_index(drop=True)
        )

        print(f"{name}: {'OK' if ordenado else 'ERROR'}")

    print("=" * 80)


In [25]:
comprobar_orden_datasets_base(
    mnq_delta_60=mnq_delta_60,
    mnq_delta_90=mnq_delta_90
)

VALIDACIÓN ORDEN TEMPORAL - DATASETS BASE
mnq_delta_60: OK
mnq_delta_90: OK


In [30]:
def crear_splits_ordenados(df, train_days, val_days, test_days):

    # asegurar orden de los días
    train_days = sorted(train_days)
    val_days   = sorted(val_days)
    test_days  = sorted(test_days)

    # filtrar
    df_train = df[df["date"].isin(train_days)].copy()
    df_valid = df[df["date"].isin(val_days)].copy()
    df_test  = df[df["date"].isin(test_days)].copy()

    # reordenar explícitamente (clave)
    df_train = df_train.sort_values(["date", "minute_of_day"])
    df_valid = df_valid.sort_values(["date", "minute_of_day"])
    df_test  = df_test.sort_values(["date", "minute_of_day"])

    return df_train, df_valid, df_test

In [31]:
mnq_delta_60_train, mnq_delta_60_valid, mnq_delta_60_test = crear_splits_ordenados(
    mnq_delta_60, train_days, val_days, test_days
)

mnq_delta_90_train, mnq_delta_90_valid, mnq_delta_90_test = crear_splits_ordenados(
    mnq_delta_90, train_days, val_days, test_days
)

In [35]:
comprobar_orden_datasets_base(
    mnq_delta_60_train=mnq_delta_60_train,
    mnq_delta_60_valid=mnq_delta_60_valid,
    mnq_delta_60_test=mnq_delta_60_test,
    mnq_delta_90_train=mnq_delta_90_train,
    mnq_delta_90_valid=mnq_delta_90_valid,
    mnq_delta_90_test=mnq_delta_90_test,
)

VALIDACIÓN ORDEN TEMPORAL - DATASETS BASE
mnq_delta_60_train: OK
mnq_delta_60_valid: OK
mnq_delta_60_test: OK
mnq_delta_90_train: OK
mnq_delta_90_valid: OK
mnq_delta_90_test: OK


In [36]:
def crear_y_validar_splits(df, train_days, val_days, test_days, nombre="dataset"):
    df_train = df[df["date"].isin(train_days)].copy()
    df_valid = df[df["date"].isin(val_days)].copy()
    df_test  = df[df["date"].isin(test_days)].copy()

    def esta_ordenado(x):
        return x[["date", "minute_of_day"]].reset_index(drop=True).equals(
            x.sort_values(["date", "minute_of_day"])[["date", "minute_of_day"]].reset_index(drop=True)
        )

    print(f"{nombre}_train:", "OK" if esta_ordenado(df_train) else "ERROR")
    print(f"{nombre}_valid:", "OK" if esta_ordenado(df_valid) else "ERROR")
    print(f"{nombre}_test :", "OK" if esta_ordenado(df_test) else "ERROR")

    return df_train, df_valid, df_test

In [37]:
mnq_delta_60_train, mnq_delta_60_valid, mnq_delta_60_test = crear_y_validar_splits(
    mnq_delta_60, train_days, val_days, test_days, nombre="mnq_delta_60"
)

mnq_delta_90_train, mnq_delta_90_valid, mnq_delta_90_test = crear_y_validar_splits(
    mnq_delta_90, train_days, val_days, test_days, nombre="mnq_delta_90"
)

mnq_delta_60_train: OK
mnq_delta_60_valid: OK
mnq_delta_60_test : OK
mnq_delta_90_train: OK
mnq_delta_90_valid: OK
mnq_delta_90_test : OK


In [34]:
def print_split_info(name, train_df, valid_df, test_df):
    print(f"\n===== {name} =====")

    # Filas
    print("Rows:")
    print("  Train:", len(train_df))
    print("  Valid:", len(valid_df))
    print("  Test :", len(test_df))

    # Días únicos
    print("Days:")
    print("  Train:", train_df["date"].nunique())
    print("  Valid:", valid_df["date"].nunique())
    print("  Test :", test_df["date"].nunique())

    # Rango temporal (index)
    print("\nTrain index:")
    print("  Desde:", train_df.index.min())
    print("  Hasta:", train_df.index.max())

    print("\nValidation index:")
    print("  Desde:", valid_df.index.min())
    print("  Hasta:", valid_df.index.max())

    print("\nTest index:")
    print("  Desde:", test_df.index.min())
    print("  Hasta:", test_df.index.max())


print_split_info("delta_60", mnq_delta_60_train, mnq_delta_60_valid, mnq_delta_60_test)
print_split_info("delta_90", mnq_delta_90_train, mnq_delta_90_valid, mnq_delta_90_test)



===== delta_60 =====
Rows:
  Train: 490146
  Valid: 104954
  Test : 105495
Days:
  Train: 906
  Valid: 194
  Test : 195

Train index:
  Desde: 2020-01-02 05:30:00-05:00
  Hasta: 2023-10-30 14:30:00-04:00

Validation index:
  Desde: 2023-10-31 05:30:00-04:00
  Hasta: 2024-08-21 14:30:00-04:00

Test index:
  Desde: 2024-08-22 05:30:00-04:00
  Hasta: 2025-06-13 14:30:00-04:00

===== delta_90 =====
Rows:
  Train: 490146
  Valid: 104954
  Test : 105495
Days:
  Train: 906
  Valid: 194
  Test : 195

Train index:
  Desde: 2020-01-02 05:30:00-05:00
  Hasta: 2023-10-30 14:30:00-04:00

Validation index:
  Desde: 2023-10-31 05:30:00-04:00
  Hasta: 2024-08-21 14:30:00-04:00

Test index:
  Desde: 2024-08-22 05:30:00-04:00
  Hasta: 2025-06-13 14:30:00-04:00


Se verifica que todos los datasets poseen una base homogénea, lo que nos va a facilitar la comparación de resultados entre etapas de entrenamiento, ajuste y evaluación. Además, la consistencia en el número de registros por día nos garantiza que los modelos reciban siempre ventanas de información con la misma extensión temporal.

# **6. Guardamos los datasets generados**


In [38]:
from pathlib import Path
import os

# Asegurar que existan los directorios
for path in [
    OUT_PARQUET_DELTA_60_TRAIN, OUT_PARQUET_DELTA_60_VALID, OUT_PARQUET_DELTA_60_TEST,
    OUT_PARQUET_DELTA_90_TRAIN, OUT_PARQUET_DELTA_90_VALID, OUT_PARQUET_DELTA_90_TEST,

]:
    path.parent.mkdir(parents=True, exist_ok=True)

# Guardar datasets
mnq_delta_60_train.to_parquet(OUT_PARQUET_DELTA_60_TRAIN)
mnq_delta_60_valid.to_parquet(OUT_PARQUET_DELTA_60_VALID)
mnq_delta_60_test.to_parquet(OUT_PARQUET_DELTA_60_TEST)

mnq_delta_90_train.to_parquet(OUT_PARQUET_DELTA_90_TRAIN)
mnq_delta_90_valid.to_parquet(OUT_PARQUET_DELTA_90_VALID)
mnq_delta_90_test.to_parquet(OUT_PARQUET_DELTA_90_TEST)

print("Splits guardados correctamente en data/splits/")


Splits guardados correctamente en data/splits/


# **7. Alineación con libro ML**


La separación de los datos se realiza respetando estrictamente el orden temporal,
de acuerdo con las buenas prácticas de *Machine Learning* para series temporales
financieras intradía.

### Principio fundamental

En problemas de series temporales **no se permiten splits aleatorios**.
El tiempo siempre fluye en una única dirección, por lo que:

- El conjunto de entrenamiento contiene únicamente información pasada.
- El conjunto de validación representa un período posterior e independiente.
- El conjunto de test simula el comportamiento futuro del modelo.

---

### Esquema de separación

- El split se realiza **por jornadas completas**, no por minutos individuales.
- Cada subconjunto contiene **días consecutivos**, sin solapamientos.
- No se mezclan observaciones de una misma jornada entre distintos splits.

De este modo, se evita cualquier fuga de información entre conjuntos.

---

### Rol de cada subconjunto

- **Train**  
  Utilizado para el entrenamiento del modelo y el ajuste de parámetros internos.

- **Validation**  
  Utilizado para:
  - selección de hiperparámetros,
  - comparación entre modelos,
  - decisiones de arquitectura.

- **Test**  
  Reservado exclusivamente para la evaluación final.
  No participa en ninguna decisión previa y se evalúa **una sola vez**.

---

### Consistencia temporal y de horizontes

- El criterio de separación es **idéntico** para los horizontes  
  **H = 60 minutos** y **H = 90 minutos**.
- La única diferencia entre ambos problemas es el horizonte del target,
  manteniéndose constantes:
  - las fechas de corte,
  - las jornadas incluidas,
  - la lógica de descarte de observaciones finales por día.

---

### Consideraciones intradía

- Los últimos minutos de cada jornada se descartan al construir los targets
  cuando no existe información suficiente hacia adelante para el horizonte definido.
- Esta regla se aplica de manera consistente en *train*, *validation* y *test*.

---

### Diagnóstico

La separación de datos implementa un **time-aware split correcto**, coherente con
el carácter secuencial del problema y adecuada para evaluar la capacidad de
generalización del modelo en un contexto intradía realista.

Este bloque **cierra el Punto 3** del proceso de *Machine Learning*.
